# Use Case 1: Loading and Cleaning the Chicago Crime Data

The goal here is to read the raw Chicago crime files, clean them up, add a few useful columns, and load everything into MySQL.

```
 CSV files ──► Pandas DataFrame ──► cleaning ──► new features ──► NumPy quality checks ──► MySQL
 (6 files)      (load + inspect)    (dates, NAs,   (Year, Month,     (% missing,             (normalized
                                     case, dupes)   DayOfWeek, Hour)  drop >50% cols)          schema)
```

This notebook produces three things that notebooks 02 to 04 rely on:
* `output/chicago_crime_clean.pkl`: the cleaned DataFrame (saved as a pickle so the datetime and bool types survive)
* the MySQL database `chicago_crime_db` with 6 tables
* `output/data_quality_report.csv`: a log of every problem I found in the data and how I handled it

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

import crime_utils as cu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
quality_log = []          # one row per data-quality finding


def log(check, rows_affected, action):
    quality_log.append({"check": check, "rows_affected": int(rows_affected), "action": action})
    print(f"[{check}] rows affected: {rows_affected:,}  ->  {action}")

## 1. Load the data into a DataFrame

`iucr_code`, `case_number` and `fbi_code` look like numbers but they're really codes (`031A`, `08B` and so on), so I read them as strings. If pandas treated them as numbers it would quietly turn `0486` into `486`, and it would fail outright on `031A`.

In [2]:
crime_path = cu.data_file("crime")

# A plain pd.read_csv(crime_path) fails on this file with:
#   "ParserError: Expected 20 fields in line 498, saw 21"
# The last two lines each open a quote that never closes, e.g. "(41.76..., -87.64...)
# (looks like the export got cut off), so the parser glues them into one bad row.
# cu.read_csv_repaired only touches records that end up with the wrong number of fields,
# so a properly quoted field that spans several lines is left alone.
raw, repaired_lines = cu.read_csv_repaired(
    crime_path, dtype={"iucr_code": str, "case_number": str, "fbi_code": str}, low_memory=False)
log("Line with unclosed quote (truncated export)", len(repaired_lines),
    f"quote closed on file lines {repaired_lines}")

# clean up column names: strip spaces, lower case, snake_case
raw.columns = raw.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

[Line with unclosed quote (truncated export)] rows affected: 2  ->  quote closed on file lines [498, 499]


### 1a. First 10 rows

In [3]:
display(raw.head(10))

,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


### 1b. Columns and data types
`df.info()` lists each column with its non-null count and dtype. A few things stand out:
* `date` and `date_of_update` came in as text (object), so they need converting to datetime.
* `arrest` and `domestic` are the strings `TRUE`/`FALSE`, not real booleans.
* `ward_no`, `community_code` and `district_code` are float64. Pandas does this whenever an integer column has missing values. I'll switch them to the nullable `Int64` type later.

In [4]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 non-null    float64
 18  l

In [5]:
schema = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "nulls": raw.isna().sum(),
    "unique_values": raw.nunique(),
    "example": raw.iloc[0],
})
display(schema)

,dtype,non_null,nulls,unique_values,example
id,int64,498,0,498,10508693
case_number,str,498,0,498,HZ250496
date,str,498,0,46,5/3/2016
block,str,498,0,475,013XX S SAWYER AVE
iucr_code,str,498,0,97,486
location_desc,str,494,4,47,APARTMENT
arrest,bool,498,0,2,True
domestic,bool,498,0,2,True
beat_num,int64,498,0,209,1022
district_code,int64,498,0,22,10


### 1c. Number of rows and columns

In [6]:
n_rows, n_cols = raw.shape
print(f"Rows: {n_rows:,}   Columns: {n_cols}")

Rows: 498   Columns: 20


### 1d. Load the 5 lookup files
These are the tables the crime data points to through foreign keys.

The files I received don't line up perfectly with the data model, so I tidy the headers while loading:
* `iucr_codes.csv.csv` has the header `IUCR, PRIMARY_TYPE␣, DESCRIPTION, INDEX CODE` (there's a trailing space after PRIMARY_TYPE and a space inside INDEX CODE). I rename these to `iucr_code, primary_type, description, index_code`.
* In `chicago_ward_offices.csv` the key column is just `ward`, so I rename it to `ward_no`.

In [7]:
RENAME = {"iucr": "iucr_code", "ward": "ward_no"}

def load_lookup(key):
    d = pd.read_csv(cu.data_file(key), dtype=str)
    d.columns = (d.columns.str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))   # 'PRIMARY_TYPE ' -> 'primary_type'
    d = d.rename(columns=RENAME)
    return d.apply(lambda s: s.str.strip())          # strip whitespace in every cell

iucr_raw = load_lookup("iucr")
community_raw = load_lookup("community")
district_raw = load_lookup("district")
ward_raw = load_lookup("ward")
beat_raw = load_lookup("beat")

for name, d in [("iucr", iucr_raw), ("community", community_raw), ("district", district_raw),
                ("ward", ward_raw), ("beat", beat_raw)]:
    print(f"{name:<10} {d.shape[0]:>4} rows x {d.shape[1]:>2} cols   {list(d.columns)[:6]} ...")

iucr        410 rows x  4 cols   ['iucr_code', 'primary_type', 'description', 'index_code'] ...
community    77 rows x  7 cols   ['community_code', 'community_name', 'population', 'area_sqmile', 'area_sqkm', 'density_per_sqmi'] ...
district     22 rows x 15 cols   ['district_code', 'district_name', 'address', 'city', 'state', 'zip'] ...
ward         58 rows x 20 cols   ['ward_no', 'alderman', 'address', 'city', 'state', 'zipcode'] ...
beat        277 rows x  4 cols   ['district', 'sector', 'beat', 'beat_num'] ...


## 2. Clean the data

I do all the cleaning on a copy called `df` and leave `raw` alone, so I can compare before and after.

### 2a. Remove duplicates
Two kinds to look for:
* Exact duplicates, i.e. the same row appearing twice (usually from exporting twice).
* Repeated `case_number`. This is the primary key so it has to be unique. If a case shows up more than once I keep the copy with the latest `date_of_update`.
* Near-duplicates: rows with the same date, block, crime code and coordinates but different case numbers. Here there are 4 such pairs (e.g. HZ251540 and HZ251541). They're most likely several victims of one incident, which get separate case numbers, so I keep them and just log that they were checked.

In [8]:
df = raw.copy()

n_exact = df.duplicated().sum()
df = df.drop_duplicates()
log("Exact duplicate rows", n_exact, "dropped")

# date_of_update is date-only in this file ('5/10/2016'), so a fixed '%m/%d/%Y %H:%M' format
# would turn every value into NaT and the sort below would do nothing. The shared parser tries
# each known format instead.
df["case_number"] = df["case_number"].str.strip().str.upper()   # " hz123" and "HZ123" are one case
df["_upd"], _ = cu.parse_mixed_dates(df["date_of_update"])
n_case_dup = df.duplicated("case_number").sum()
df = (df.sort_values("_upd").drop_duplicates("case_number", keep="last")
        .drop(columns="_upd").sort_index())
log("Duplicate case_number (PK)", n_case_dup, "kept the most recently updated record")

n_no_case = df["case_number"].isna().sum()
df = df[df["case_number"].notna()]
log("Missing case_number", n_no_case, "dropped (cannot be stored without a primary key)")

# Near-duplicates: same date, block, crime code and coordinates but a different case_number.
# These are most likely several victims of one incident, which Chicago records as separate cases.
NEAR_KEYS = ["date", "block", "iucr_code", "latitude", "longitude"]
near = df.duplicated(NEAR_KEYS, keep=False) & df["latitude"].notna()
display(df.loc[near, ["case_number"] + NEAR_KEYS].sort_values(NEAR_KEYS))
log("Same date/block/code/coordinates, different case_number", near.sum(),
    f"kept ({near.sum() // 2} pairs; most likely several victims of one incident)")

[Exact duplicate rows] rows affected: 0  ->  dropped
[Duplicate case_number (PK)] rows affected: 0  ->  kept the most recently updated record
[Missing case_number] rows affected: 0  ->  dropped (cannot be stored without a primary key)


,case_number,date,block,iucr_code,latitude,longitude
435,HZ251429,5/4/2016,011XX W JACKSON BLVD,860,41.877822,-87.655181
453,HZ251461,5/4/2016,011XX W JACKSON BLVD,860,41.877822,-87.655181
444,HZ251475,5/4/2016,052XX S LAKE PARK AVE,890,41.800483,-87.587415
456,HZ251570,5/4/2016,052XX S LAKE PARK AVE,890,41.800483,-87.587415
496,HZ251541,5/4/2016,071XX S MORGAN ST,1310,41.764155,-87.649243
497,HZ251540,5/4/2016,071XX S MORGAN ST,1310,41.764155,-87.649243
436,HZ251358,5/4/2016,076XX S CICERO AVE,860,41.754593,-87.741529
437,HZ251410,5/4/2016,076XX S CICERO AVE,860,41.754593,-87.741529


[Same date/block/code/coordinates, different case_number] rows affected: 8  ->  kept (4 pairs; most likely several victims of one incident)


### 2b. Convert `date` to datetime

The expected format is `5/3/2016 23:40`, i.e. `%m/%d/%Y %H:%M`. Exports often contain a mix of formats though, and a single `pd.to_datetime` call with one format would just turn everything else into `NaT` without saying anything. So instead I try a short list of known formats one after another and note which one worked for each row. Whatever doesn't match any of them is a real problem worth logging. The parser lives in `crime_utils.py` (`cu.parse_mixed_dates`) so the duplicate check above can use it too, and so it can be tested.

In this file every `date` is just `m/d/yyyy` with no time, and `arrest`/`domestic` are upper-case `TRUE`/`FALSE`. That's what you get when a CSV has been opened and saved in Excel. The Chicago data portal export normally includes a time, so the times were most likely lost before I received the file. Getting the raw export would make the hourly analysis in notebook 03 possible.

In [9]:
# cu.DATE_FORMATS lists the formats in the order they are tried:
for f in cu.DATE_FORMATS:
    print("  ", f)

df["date_parsed"], df["date_format"] = cu.parse_mixed_dates(df["date"])
print("Which date formats were found:")
display(df["date_format"].value_counts(dropna=False).rename("rows").to_frame())

   %m/%d/%Y %H:%M
   %m/%d/%Y %I:%M:%S %p
   %m/%d/%Y %H:%M:%S
   %Y-%m-%d %H:%M:%S
   %Y-%m-%d %H:%M
   %m/%d/%Y
Which date formats were found:


,rows
date_format,
%m/%d/%Y,498


In [10]:
bad_dates = df[df["date_parsed"].isna()]
print("Examples of values that could not be parsed:")
display(bad_dates["date"].value_counts(dropna=False).head(10).rename("rows").to_frame())

dominant = df["date_format"].mode().iloc[0]
n_alt = (df["date_format"].notna() & (df["date_format"] != dominant)).sum()
print(f"Dominant format: {dominant}")
log("Date in a different format from the rest", n_alt, "parsed with the matching format")

date_only = df["date_format"].isin(cu.DATE_ONLY_FORMATS)
log("Date has no time of day (date only)", date_only.sum(),
    "hour stored as NULL (unknown), not as 00:00")
log("Date missing / unparseable", len(bad_dates), "dropped (a crime with no date can't be placed in time)")
df = df[df["date_parsed"].notna()].copy()

Examples of values that could not be parsed:


,rows
date,


Dominant format: %m/%d/%Y
[Date in a different format from the rest] rows affected: 0  ->  parsed with the matching format
[Date has no time of day (date only)] rows affected: 498  ->  hour stored as NULL (unknown), not as 00:00
[Date missing / unparseable] rows affected: 0  ->  dropped (a crime with no date can't be placed in time)


A date can parse fine and still be wrong, so I also check for:
* crime dates in the future
* a crime date that comes after its `date_of_update` (a record can't be updated before the crime happened)
* a `year` column that doesn't agree with the year in `date`

In [11]:
df["date"] = df.pop("date_parsed")
df["_has_time"] = ~df["date_format"].isin(cu.DATE_ONLY_FORMATS)   # used for 'hour' in step 3
df["date_of_update"], _ = cu.parse_mixed_dates(df["date_of_update"])
df = df.drop(columns="date_format")

today = pd.Timestamp.today().normalize()
future = df["date"] > today
log("Crime date in the future", future.sum(), "dropped")
df = df[~future]

upd_before = df["date_of_update"].notna() & (df["date_of_update"] < df["date"])
log("date_of_update earlier than date", upd_before.sum(), "flagged only (kept)")

year_col = pd.to_numeric(df["year"], errors="coerce")
year_mismatch = year_col != df["date"].dt.year
log("year column disagrees with date", year_mismatch.sum(), "year rebuilt from date")
df["year"] = df["date"].dt.year

print(f"\nDate range after cleaning: {df['date'].min()}  →  {df['date'].max()}")

[Crime date in the future] rows affected: 0  ->  dropped
[date_of_update earlier than date] rows affected: 0  ->  flagged only (kept)
[year column disagrees with date] rows affected: 0  ->  year rebuilt from date

Date range after cleaning: 2012-01-01 00:00:00  →  2016-05-04 00:00:00


### 2c. Tidy up the text columns (whitespace and case)

`"APARTMENT"`, `"Apartment"` and `"  apartment "` mean the same thing, but pandas would count them as three different values. I strip spaces, collapse double spaces and upper-case every text column.

The IUCR codes need one more fix. The official list uses 4 characters (`0486`), but a lot of files drop the leading zero (`486`). I pad both the crime data and the lookup table with `zfill(4)` so the join works.

In [12]:
def standardize_text(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\s+", " ", regex=True).str.upper())

before = df["location_desc"].nunique()
for col in ["block", "location_desc", "fbi_code", "iucr_code", "case_number"]:
    df[col] = standardize_text(df[col])
print(f"location_desc distinct values: {before} before → {df['location_desc'].nunique()} after")

df["iucr_code"] = df["iucr_code"].str.zfill(4)
iucr = iucr_raw.copy()
iucr["iucr_code"] = standardize_text(iucr["iucr_code"]).str.zfill(4)
for col in ["primary_type", "description", "index_code"]:
    iucr[col] = standardize_text(iucr[col])
iucr = iucr.drop_duplicates("iucr_code")

location_desc distinct values: 47 before → 47 after


`arrest` and `domestic` come in as text in various spellings (`TRUE`, `true`, `FALSE` etc.). I map all the usual variants to proper `bool` values.

In [13]:
BOOL_MAP = {"TRUE": True, "T": True, "Y": True, "YES": True, "1": True, "1.0": True,
            "FALSE": False, "F": False, "N": False, "NO": False, "0": False, "0.0": False}

for col in ["arrest", "domestic"]:
    txt = standardize_text(df[col])
    mapped = txt.map(BOOL_MAP)
    unmapped = txt.notna() & mapped.isna()
    if unmapped.any():
        print(f"{col}: unexpected values {txt[unmapped].unique()[:5]}")
    log(f"{col} missing/unrecognised", mapped.isna().sum(),
        "set to False (no record of an arrest/domestic flag)")
    df[col] = mapped.fillna(False).astype(bool)

print(df[["arrest", "domestic"]].dtypes)

[arrest missing/unrecognised] rows affected: 0  ->  set to False (no record of an arrest/domestic flag)
[domestic missing/unrecognised] rows affected: 0  ->  set to False (no record of an arrest/domestic flag)
arrest      bool
domestic    bool
dtype: object


The numeric code columns (`beat_num`, `district_code`, `ward_no`, `community_code`) and the coordinates go through `pd.to_numeric(errors="coerce")`. For the codes I use the nullable `Int64` type, so a missing ward shows up as `<NA>` and the rest of the column stays as integers.

In [14]:
for col in ["beat_num", "district_code", "ward_no", "community_code", "x_coordinate", "y_coordinate", "id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").round().astype("Int64")
for col in ["latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

Next I join the IUCR lookup to get `primary_type` (the crime category) and `description`. The main file only has the code; the category name is in `iucr_codes.csv`.

In [15]:
if "primary_type" in df.columns:          # some versions of the file already include it
    df = df.drop(columns=[c for c in ["primary_type", "description"] if c in df.columns])
df = df.merge(iucr[["iucr_code", "primary_type", "description"]], on="iucr_code", how="left")

unknown_iucr = df["primary_type"].isna()
print("IUCR codes not found in lookup:", df.loc[unknown_iucr, "iucr_code"].unique())
log("IUCR code not in lookup table", unknown_iucr.sum(), "primary_type = 'UNKNOWN'")
df["primary_type"] = df["primary_type"].fillna("UNKNOWN")
df["description"] = df["description"].fillna("UNKNOWN")

IUCR codes not found in lookup: <StringArray>
[]
Length: 0, dtype: string
[IUCR code not in lookup table] rows affected: 0  ->  primary_type = 'UNKNOWN'


### 2d. Missing values in the key fields

There isn't one rule that fits every column, so I decided case by case. The general idea: don't fill in anything I can't back up.

| Field | What I did | Reason |
|---|---|---|
| `case_number`, `date` | drop the row | done already; it's the primary key, and a crime with no date isn't usable |
| `iucr_code` | drop the row | without a code there's no crime category |
| `district_code` | fill it from `beat_num` using the beat lookup | each beat sits in exactly one district, so this isn't a guess. If a beat isn't in the lookup I fall back to `beat // 100`, which is a guess, so it's logged separately |
| `location_desc` | fill with `"UNKNOWN"` | keeps the row and makes the gap visible in charts |
| `ward_no`, `community_code` | leave as `NULL` | guessing an area would skew the area-level numbers; could be filled later with a spatial join on lat/long |
| `latitude`/`longitude`/`x`/`y` | leave as `NULL` | not needed for counting, and I don't want to make up locations |

In [16]:
key_fields = ["case_number", "date", "iucr_code", "primary_type", "arrest", "domestic",
              "location_desc", "beat_num", "district_code", "ward_no", "community_code",
              "latitude", "longitude"]
missing_before = df[key_fields].isna().sum().rename("missing_before")

# iucr_code
n = df["iucr_code"].isna().sum()
df = df[df["iucr_code"].notna()]
log("Missing iucr_code", n, "dropped")

# district from beat: the beat lookup gives the real district. If the beat isn't in the lookup,
# fall back to beat // 100 (how Chicago numbers its beats). That fallback is an educated guess,
# so it is logged separately.
beat_lu = beat_raw.assign(beat_num=pd.to_numeric(beat_raw["beat_num"], errors="coerce"),
                          district=pd.to_numeric(beat_raw["district"], errors="coerce"))
beat_to_district = beat_lu.drop_duplicates("beat_num").set_index("beat_num")["district"]
miss_d = df["district_code"].isna() & df["beat_num"].notna()
from_lookup = df.loc[miss_d, "beat_num"].map(beat_to_district)
guessed = from_lookup.isna()
derived = from_lookup.fillna(df.loc[miss_d, "beat_num"] // 100)
df.loc[miss_d, "district_code"] = derived.astype("Int64")
log("Missing district_code (filled from beat lookup)", (~guessed).sum(), "filled from police_beat_info")
log("Missing district_code (beat not in lookup)", guessed.sum(), "guessed as beat_num // 100")

n = df["location_desc"].isna().sum()
df["location_desc"] = df["location_desc"].fillna("UNKNOWN")
log("Missing location_desc", n, "filled with 'UNKNOWN'")

for col in ["ward_no", "community_code"]:
    log(f"Missing {col}", df[col].isna().sum(), "kept as NULL")

missing_after = df[key_fields].isna().sum().rename("missing_after")
display(pd.concat([missing_before, missing_after], axis=1))

[Missing iucr_code] rows affected: 0  ->  dropped
[Missing district_code (filled from beat lookup)] rows affected: 0  ->  filled from police_beat_info
[Missing district_code (beat not in lookup)] rows affected: 0  ->  guessed as beat_num // 100
[Missing location_desc] rows affected: 4  ->  filled with 'UNKNOWN'
[Missing ward_no] rows affected: 0  ->  kept as NULL
[Missing community_code] rows affected: 0  ->  kept as NULL


,missing_before,missing_after
case_number,0,0
date,0,0
iucr_code,0,0
primary_type,0,0
arrest,0,0
domestic,0,0
location_desc,4,0
beat_num,0,0
district_code,0,0
ward_no,0,0


### 2e. Coordinates outside Chicago
Chicago sits roughly between latitude 41.64 and 42.03, and longitude -87.95 and -87.52.
* A positive longitude of about the right size (say `87.70`) is almost certainly a missing minus sign, so I flip it.
* Anything else outside that box (for example `36.6, -91.7`, which is in Missouri) is just wrong, so I set it to `NULL`.

In [17]:
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 41.64, 42.03, -87.95, -87.52

sign_flip = df["longitude"].between(-LON_MAX, -LON_MIN)
df.loc[sign_flip, "longitude"] = -df.loc[sign_flip, "longitude"]
log("Longitude with missing minus sign", sign_flip.sum(), "sign corrected")

has_coords = df["latitude"].notna() & df["longitude"].notna()
outside = has_coords & ~(df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX))
print(df.loc[outside, ["case_number", "latitude", "longitude"]].head())
log("Coordinates outside Chicago", outside.sum(), "lat/long/x/y set to NULL")
df.loc[outside, ["latitude", "longitude", "x_coordinate", "y_coordinate"]] = np.nan

# rebuild the 'location' text using the fixed coordinates
df["location"] = np.where(df["latitude"].notna(),
                          "(" + df["latitude"].round(8).astype(str) + ", " + df["longitude"].round(8).astype(str) + ")",
                          None)

[Longitude with missing minus sign] rows affected: 0  ->  sign corrected
Empty DataFrame
Columns: [case_number, latitude, longitude]
Index: []
[Coordinates outside Chicago] rows affected: 0  ->  lat/long/x/y set to NULL


## 3. Add new columns
The `.dt` accessor only works on datetime columns, which is why the date conversion in 2b had to happen first. I also add `Hour` here since Use Case 3 uses it.

`hour` is only filled in when the source date had a time. A date-only value like `5/3/2016` would give `hour = 0`, which looks like a real midnight crime. That would be inventing data, so those rows get `NULL` instead (and in this file that's every row).

In [18]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
# hour only exists when the source had a time; otherwise it is unknown (NULL), not midnight
df["hour"] = df["date"].dt.hour.where(df.pop("_has_time")).astype("Int64")

display(df[["date", "year", "month", "day_of_week", "hour"]].head())

,date,year,month,day_of_week,hour
0,2016-05-03,2016,5,Tuesday,<NA>
1,2016-05-03,2016,5,Tuesday,<NA>
2,2016-05-03,2016,5,Tuesday,<NA>
3,2016-05-03,2016,5,Tuesday,<NA>
4,2016-05-03,2016,5,Tuesday,<NA>


## 4. Percentage of missing values per column (NumPy)

`df.isna().to_numpy()` gives a 2-D array of True/False values, where True means missing. True counts as 1 and False as 0, so taking `np.mean(..., axis=0)` down each column gives the fraction of missing values in that column. I run it on both the raw data and the cleaned data to compare.

In [19]:
def missing_pct_numpy(frame):
    mask = frame.isna().to_numpy()                    # shape (rows, cols), dtype=bool
    pct = np.mean(mask, axis=0) * 100                 # column-wise mean -> fraction missing
    return pd.Series(np.round(pct, 3), index=frame.columns)

missing_report = pd.DataFrame({
    "raw_missing_%": missing_pct_numpy(raw),
    "clean_missing_%": missing_pct_numpy(df.reindex(columns=raw.columns)),
}).sort_values("raw_missing_%", ascending=False)
display(missing_report)

,raw_missing_%,clean_missing_%
location,5.823,5.823
y_coordinate,5.823,5.823
longitude,5.823,5.823
latitude,5.823,5.823
x_coordinate,5.823,5.823
location_desc,0.803,0.000
case_number,0.000,0.000
id,0.000,0.000
block,0.000,0.000
date,0.000,0.000


### Dropping columns with more than 50% missing
This is easy to do: `df.columns[pct > 50]` finds them and `df.drop(columns=...)` removes them. I do keep a short list of key columns protected though. If `date` or `primary_type` were 60% empty, dropping them would wreck the analysis, and that's something to look into rather than delete.

In [20]:
THRESHOLD = 50
# 'hour' is protected too: it is empty because the source has no times, and keeping the column
# (all NULL) records that, and keeps the table layout the same for a file that does have times.
PROTECTED = {"id", "case_number", "date", "iucr_code", "primary_type", "arrest", "domestic", "hour"}

pct_now = missing_pct_numpy(df)
over = pct_now[pct_now > THRESHOLD].index.tolist()
to_drop = [c for c in over if c not in PROTECTED]
blocked = [c for c in over if c in PROTECTED]

print(f"Columns with > {THRESHOLD}% missing: {over or 'none'}")
if blocked:
    print(f"NOT dropped (protected key fields, investigate instead): {blocked}")
df = df.drop(columns=to_drop)
print(f"Dropped: {to_drop or 'nothing'} | Highest missing % in any column: "
      f"{pct_now.max():.2f}% ({pct_now.idxmax()})")

Columns with > 50% missing: ['hour']
NOT dropped (protected key fields, investigate instead): ['hour']
Dropped: nothing | Highest missing % in any column: 100.00% (hour)


## 5. Load the cleaned data into MySQL

### 5a. Schema
The schema is in `sql/01_create_schema.sql` and follows the data model, which is a star schema:

* `chicago_crime` is the fact table, with one row per incident.
* `iucr`, `city_community`, `district_ps_info`, `ward_office` and `police_beat_info` are the lookup tables, connected with foreign keys.
* `primary_type` is only stored in `iucr`. Each fact lives in one place and we join when we need it.
* `year`, `month`, `day_of_week` and `hour` are stored directly on the fact table. That's a bit of denormalisation on purpose, so the reporting queries don't have to recalculate them every time.
* There are indexes on `year`, `(year, month)` and `community_code` to speed up the Use Case 4 queries.

### 5b. Writing with SQLAlchemy and PyMySQL
The order matters here. The lookup tables have to go in first and the fact table last, otherwise MySQL rejects crime rows that point to a lookup row that isn't there yet.

The load cell empties the tables before writing (crimes first, then lookups), so it can be run again on its own without duplicate-key errors.

In [21]:
from sqlalchemy import text

server = cu.get_engine(with_db=False)
with server.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {cu.DB['database']} "
                      "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = cu.get_engine()
n_stmt = cu.run_sql_file(engine, cu.SQL_DIR / "01_create_schema.sql")
print(f"Schema created ({n_stmt} statements)")

Schema created (14 statements)


Before loading, the lookup tables need a bit of prep: text like `"55,062"` has to become a number, and every code used in the crime table has to exist in its lookup.

In [22]:
def to_num(s, kind=float):
    v = pd.to_numeric(s.astype("string").str.replace(",", "", regex=False), errors="coerce")
    return v.round().astype("Int64") if kind is int else v

community = community_raw.copy()
community["community_code"] = to_num(community["community_code"], int)
community["population"] = to_num(community["population"], int)
for c in ["area_sqmile", "area_sqkm", "density_per_sqmi", "density_per_sqkm"]:
    community[c] = to_num(community[c])
community["community_name"] = standardize_text(community["community_name"])
community = community.dropna(subset=["community_code"]).drop_duplicates("community_code")

district = district_raw.copy()
district["district_code"] = to_num(district["district_code"], int)
for c in ["x_coordinate", "y_coordinate", "latitude", "longitude"]:
    district[c] = to_num(district[c])
district = district.dropna(subset=["district_code"]).drop_duplicates("district_code")

ward = ward_raw.copy()
ward["ward_no"] = to_num(ward["ward_no"], int)
ward = ward.dropna(subset=["ward_no"])
# A few wards are listed twice: an old office record, plus a newer one added at the bottom of the
# file (e.g. ward 30 Reboyras -> Cruz, ward 6 Sawyer -> Hall). Keep the last one.
dup_wards = sorted(ward.loc[ward["ward_no"].duplicated(keep=False), "ward_no"].unique().tolist())
log("Duplicate ward_no in ward offices file", ward["ward_no"].duplicated().sum(),
    f"kept the last (newest) record for wards {dup_wards}; newest is not the office "
    "in place on the crime date (see note)")
ward = ward.drop_duplicates("ward_no", keep="last")

beat = beat_raw.copy()
for c in ["beat_num", "district", "sector", "beat"]:
    beat[c] = to_num(beat[c], int)
beat = beat.dropna(subset=["beat_num"])
log("Duplicate beat_num in police beat file", beat["beat_num"].duplicated().sum(),
    "identical rows – duplicates dropped")
beat = beat.drop_duplicates("beat_num")

# every IUCR code in the crime data has to exist in iucr (foreign key), so add placeholders for any unknown ones
missing_codes = sorted(set(df["iucr_code"]) - set(iucr["iucr_code"]))
iucr_db = pd.concat([iucr, pd.DataFrame({"iucr_code": missing_codes, "primary_type": "UNKNOWN",
                                         "description": "CODE NOT IN LOOKUP", "index_code": None})],
                    ignore_index=True)
print("Placeholder IUCR codes added:", missing_codes)

[Duplicate ward_no in ward offices file] rows affected: 8  ->  kept the last (newest) record for wards [2, 6, 20, 27, 30, 31, 35, 45]; newest is not the office in place on the crime date (see note)
[Duplicate beat_num in police beat file] rows affected: 2  ->  identical rows – duplicates dropped
Placeholder IUCR codes added: []


**About the ward offices.** Eight wards appear twice in the ward file, and I keep the last (newest) record. That isn't quite right for this data, though. Several of the newer aldermen (Cruz in ward 30, Hall in ward 6, Quezada in ward 35) took office in 2023, while the crimes are from 2016. "Newest" isn't the same as "the office in place when the crime happened". This is the classic slowly changing dimension problem. The textbook fix (SCD Type 2) keeps every office record with valid-from and valid-to dates and joins each crime to the record that was valid on its date. The file has no dates for the terms, so I can't do that here. Ward 27's two rows are the same person spelled two ways (Burnett, Jr., Walter / Burnett, Walter R.).

If a crime points to a beat, ward, district or community that isn't in the lookup file, MySQL will refuse the row. I set those references to `NULL` and keep a count of how many there were.

In [23]:
fk_checks = {"beat_num": beat["beat_num"], "district_code": district["district_code"],
             "ward_no": ward["ward_no"], "community_code": community["community_code"]}
for col, valid in fk_checks.items():
    orphan = df[col].notna() & ~df[col].isin(valid.dropna())
    if orphan.any():
        df.loc[orphan, col] = pd.NA
    log(f"Orphan FK {col}", orphan.sum(), "set to NULL")

[Orphan FK beat_num] rows affected: 0  ->  set to NULL
[Orphan FK district_code] rows affected: 0  ->  set to NULL
[Orphan FK ward_no] rows affected: 0  ->  set to NULL
[Orphan FK community_code] rows affected: 0  ->  set to NULL


In [24]:
CRIME_COLS = ["id", "case_number", "date", "block", "iucr_code", "location_desc", "arrest",
              "domestic", "beat_num", "district_code", "ward_no", "community_code", "fbi_code",
              "x_coordinate", "y_coordinate", "year", "month", "day_of_week", "hour",
              "date_of_update", "latitude", "longitude", "location"]

crime_db = df[[c for c in CRIME_COLS if c in df.columns]].copy()
crime_db["arrest"] = crime_db["arrest"].astype(int)
crime_db["domestic"] = crime_db["domestic"].astype(int)
crime_db = crime_db.astype(object).where(crime_db.notna(), None)   # NaN/NA -> SQL NULL

loads = [("iucr", iucr_db), ("city_community", community), ("district_ps_info", district),
         ("ward_office", ward), ("police_beat_info", beat), ("chicago_crime", crime_db)]

# Empty the tables first so this cell can be run again without duplicate-key errors.
# Crimes go first, otherwise the foreign keys stop the lookups from being emptied.
with engine.begin() as conn:
    for table, _ in reversed(loads):
        conn.execute(text(f"DELETE FROM {table}"))

for table, frame in loads:
    cols = pd.read_sql(text(f"SELECT * FROM {table} LIMIT 0"), engine).columns
    frame = frame[[c for c in frame.columns if c in cols]]
    frame.to_sql(table, engine, if_exists="append", index=False, chunksize=5000, method="multi")
    print(f"Loaded {len(frame):>7,} rows into {table}")

Loaded     410 rows into iucr
Loaded      77 rows into city_community
Loaded      22 rows into district_ps_info
Loaded      50 rows into ward_office
Loaded     275 rows into police_beat_info


Loaded     498 rows into chicago_crime


Quick check that the row counts in MySQL match the DataFrames.

In [25]:
counts = pd.read_sql(text("""
    SELECT 'chicago_crime' AS table_name, COUNT(*) AS rows_in_mysql FROM chicago_crime
    UNION ALL SELECT 'iucr', COUNT(*) FROM iucr
    UNION ALL SELECT 'city_community', COUNT(*) FROM city_community
    UNION ALL SELECT 'district_ps_info', COUNT(*) FROM district_ps_info
    UNION ALL SELECT 'ward_office', COUNT(*) FROM ward_office
    UNION ALL SELECT 'police_beat_info', COUNT(*) FROM police_beat_info"""), engine)
display(counts)
assert counts.loc[counts.table_name == "chicago_crime", "rows_in_mysql"].item() == len(df)
print("Row counts match")

,table_name,rows_in_mysql
0,chicago_crime,498
1,iucr,410
2,city_community,77
3,district_ps_info,22
4,ward_office,50
5,police_beat_info,275


Row counts match


## Questions

### Q1. How many unique crime types are in the dataset?
I'm taking "crime type" to mean `primary_type` (the IUCR category). The more detailed IUCR codes are shown as well.

In [26]:
known = df.loc[df["primary_type"] != "UNKNOWN", "primary_type"]
print(f"Unique crime types (primary_type): {known.nunique()}")
print(f"Unique IUCR codes (sub-types):     {df['iucr_code'].nunique()}")
print(f"Rows with an unrecognised code:    {(df['primary_type'] == 'UNKNOWN').sum()}")
print("\nCrime types:", ", ".join(sorted(known.unique())))

Unique crime types (primary_type): 23
Unique IUCR codes (sub-types):     97
Rows with an unrecognised code:    0

Crime types: ASSAULT, BATTERY, BURGLARY, CRIMINAL DAMAGE, CRIMINAL SEXUAL ASSAULT, CRIMINAL TRESPASS, DECEPTIVE PRACTICE, GAMBLING, HOMICIDE, INTERFERENCE WITH PUBLIC OFFICER, KIDNAPPING, MOTOR VEHICLE THEFT, NARCOTICS, OBSCENITY, OFFENSE INVOLVING CHILDREN, OTHER OFFENSE, PROSTITUTION, PUBLIC PEACE VIOLATION, ROBBERY, SEX OFFENSE, STALKING, THEFT, WEAPONS VIOLATION


### Q2. Are there any problems with the date formats, or obvious outliers?
The data quality log below covers this: each issue, how many rows it affected, and what I did about it. I also check the numeric columns for values that don't make sense.

In [27]:
quality = pd.DataFrame(quality_log)
display(quality[quality["rows_affected"] > 0])

display(df[["year", "month", "hour", "beat_num", "district_code", "ward_no", "community_code",
            "latitude", "longitude"]].describe().T[["min", "max", "mean"]])

,check,rows_affected,action
0,Line with unclosed quote (truncated export),2,"quote closed on file lines [498, 499]"
4,"Same date/block/code/coordinates, different ca...",8,kept (4 pairs; most likely several victims of ...
6,Date has no time of day (date only),498,"hour stored as NULL (unknown), not as 00:00"
17,Missing location_desc,4,filled with 'UNKNOWN'
22,Duplicate ward_no in ward offices file,8,"kept the last (newest) record for wards [2, 6,..."
23,Duplicate beat_num in police beat file,2,identical rows – duplicates dropped


,min,max,mean
year,2012.0,2016.0,2015.965863
month,1.0,11.0,4.821285
hour,<NA>,<NA>,<NA>
beat_num,111.0,2535.0,1115.927711
district_code,1.0,25.0,10.925703
ward_no,1.0,50.0,22.51004
community_code,1.0,77.0,37.11245
latitude,41.65325,42.019399,41.842657
longitude,-87.839659,-87.535758,-87.671066


Even if every value is formatted correctly, the data can still be lopsided. Here I look at how the crimes are spread across dates.

In [28]:
by_day = df["date"].dt.date.value_counts()
top2_share = by_day.head(2).sum() / len(df) * 100
print(f"Date range: {df['date'].min():%d %b %Y} → {df['date'].max():%d %b %Y}")
print(f"Distinct dates: {by_day.size} | busiest 2 dates hold {top2_share:.1f}% of all crimes:")
display(by_day.head(5).rename("crimes").to_frame())
display(df["year"].value_counts().sort_index().rename("crimes").to_frame().T)
print(f"Rows with a real time of day: {(df['date'].dt.time != pd.Timestamp(0).time()).sum()}")
if top2_share > 50:
    print("\nNote: the extract is concentrated on a few days. It looks like the records reported or "
          "updated in early May 2016 (date_of_update is May 2016 for every row). Older dates are crimes "
          "that were reported late, not a full history, so keep this in mind for any year/month trend.")

Date range: 01 Jan 2012 → 04 May 2016
Distinct dates: 46 | busiest 2 dates hold 89.0% of all crimes:


,crimes
date,
2016-05-04,298
2016-05-03,145
2016-04-01,4
2015-01-01,2
2016-01-01,2


year,2012,2013,2014,2015,2016
crimes,2,1,1,4,490


Rows with a real time of day: 0

Note: the extract is concentrated on a few days. It looks like the records reported or updated in early May 2016 (date_of_update is May 2016 for every row). Older dates are crimes that were reported late, not a full history, so keep this in mind for any year/month trend.


## Save the outputs for the other notebooks

In [29]:
df.to_pickle(cu.CLEAN_CRIME_FILE)
community.to_pickle(cu.OUTPUT_DIR / "community_lookup.pkl")
quality.to_csv(cu.OUTPUT_DIR / "data_quality_report.csv", index=False)
print(f"Saved {len(df):,} clean rows → {cu.CLEAN_CRIME_FILE.name}")
print(f"Raw rows: {len(raw):,}  →  clean rows: {len(df):,}  "
      f"({100 * (1 - len(df) / len(raw)):.2f}% removed)")

Saved 498 clean rows → chicago_crime_clean.pkl
Raw rows: 498  →  clean rows: 498  (0.00% removed)
